In [16]:
import pandas as pd
from sqlalchemy import create_engine
import pymysql

import warnings
# Suppress all warnings
warnings.filterwarnings("ignore")

# MySQL connection details
host = 'localhost'
user = 'towdevuser'
password = 'Dev703'
database = 'timelineofwealth'

# # Industry codes
# energy_industry_code = '10'
# material_industry_code = '15'
# industrial_industry_code = '20'
# discretionary_industry_code = '25'
# staples_industry_code = '30'
# healthcare_industry_code = '35'
# financials_industry_code = '40'
# technology_industry_code = '45'
# telecom_industry_code = '50'
# utility_industry_code = '55'
# all_industry = ''

# Parameters for filtering
# subindustryid = all_industry
subindustry = 'all_industry'

# large_cap_min_range = 50000
# large_cap_max_range = 5000000

# mid_cap_min_range = 20000
# mid_cap_max_range = 50000

# small_cap_min_range = 2000
# small_cap_max_range = 20000

# micro_cap_min_range = 100
# micro_cap_max_range = 2000

# Set market cap range
# mcap_min_range = micro_cap_min_range
# mcap_max_range = large_cap_max_range
cap = 'all' #'all', 'small', 'large', 'mid', 'micro'

# Load annual data
query_annual = """
SELECT 
    a.date,
    a.ticker,
    -- P&L Data
    a.sales,
    a.expenses,
    a.operating_profit,
    a.other_income,
    a.depreciation,
    a.interest,
    a.profit_before_tax,
    a.tax,
    a.net_profit,
    a.eps,
    a.pe,
    a.price,
    a.dummy1 AS market_cap,
    a.ratios AS ebitm,
    a.dividend_payout,
    a.opm,
    a.npm,
    a.re AS roic,
    a.tax_rate,
    a.sales_g,
    a.ebitda_g,
    a.pat_g,
    a.non_op_inc_g,
    a.debt_to_capital,
    a.ppe_to_sales,
    a.dep_to_ppe,
    a.non_op_inc_to_invst,
    a.noplat,
    a.capex AS capex1,
    a.fcff,
    a.sales_g_3yr,
    a.sales_g_5yr,
    a.sales_g_10yr,
    a.ebitda_g_3yr,
    a.ebitda_g_5yr,
    a.ebitda_g_10yr,
    a.avg_opm_3yr,
    a.avg_opm_5yr,
    a.avg_opm_10yr,
    a.avg_opm,
    a.opm_min,
    a.opm_max,
    a.avg_roic_3yr,
    a.avg_roic_5yr,
    a.avg_roic_10yr,
    a.avg_roic,
    a.roic_min,
    a.roic_max,
    a.avg_ppe_to_sales,
    a.avg_dep_to_ppe,
    -- Balance Sheet Data
    b.equity_share_capital,
    b.reserves,
    b.borrowings,
    b.other_liabilities,
    b.total_equity_and_debt,
    b.dummy1 AS other_non_curr_asset,
    b.net_block,
    b.capital_work_in_progress,
    b.investments,
    b.other_assets,
    b.total_assets,
    b.capex as capex2,
    b.working_capital,
    b.debtors,
    b.inventory,
    b.dummy2 AS excess_cash,
    b.debtor_days,
    b.inventory_turnover,
    b.dummy3 AS op_cash,
    (b.dummy2 + b.dummy3) AS total_cash,
    b.return_on_equity,
    b.return_on_capital_emp,
    -- Cash Flow Data
    c.cash_from_operating_activity,
    c.cash_from_investing_activity,
    c.cash_from_financing_activity,
    c.net_cashflow
FROM 
    stock_pnl a
JOIN 
    stock_balancesheet b ON a.ticker = b.ticker AND a.date = b.date
JOIN 
    stock_cashflow c ON a.ticker = c.ticker AND a.date = c.date
WHERE 
    a.date > '2005-01-01' 
ORDER BY 
    a.date, a.ticker;
"""

# Load quarterly data
query_quarterly = """
SELECT 
    date,
    ticker,
    sales,
    expenses,
    operating_profit,
    other_income,
    depreciation,
    interest,
    profit_before_tax,
    tax,
    net_profit,
    ttm_sales_g,
    ttm_ebitda_g,
    ttm_opm,
    noplat,
    mcap,
    price,
    result_date
FROM 
    stock_quarter
WHERE 
    date > '2005-01-01'
ORDER BY 
    date, ticker;
"""
# Establish a connection to the database
try:
    connection = pymysql.connect(host=host, user=user, password=password, database=database)
    # Execute the query with parameters
    subindustryid_param = f"{subindustryid}%" if subindustryid else "%"
    df_annual = pd.read_sql(query_annual, connection)
    df_quarterly = pd.read_sql(query_quarterly, connection)    
except pymysql.MySQLError as e:
    print(f"Error: {e}")
finally:
    # Close the connection
    if connection:
        connection.close()

df_annual['date'] = pd.to_datetime(df_annual['date'])
df_quarterly['date'] = pd.to_datetime(df_quarterly['date'])
df_quarterly['result_date'] = pd.to_datetime(df_quarterly['result_date'])

# Filter result_date where it's greater than '2005-01-01'
df_quarterly['result_date'] = df_quarterly['result_date'].apply(
    lambda x: x if pd.notnull(x) and x > pd.Timestamp('2005-01-01') else pd.NaT
)

# Map TTM fields to appropriate columns
df_quarterly['sales_g'] = df_quarterly['ttm_sales_g']
df_quarterly['ebitda_g'] = df_quarterly['ttm_ebitda_g']
df_quarterly['opm'] = df_quarterly['ttm_opm']
df_quarterly['market_cap'] = df_quarterly['mcap']

# Calculate TTM values for selected columns
def calculate_ttm_inplace(df, columns):
    df_ttm = df.copy()
    for col in columns:
        df_ttm[col] = (
            df_ttm.groupby('ticker')[col]
            .rolling(window=4, min_periods=4)
            .sum()
            .reset_index(level=0, drop=True)
        )
    return df_ttm

# Columns for which to calculate TTM
ttm_columns = [
    'sales', 'expenses', 'operating_profit', 'other_income',
    'depreciation', 'interest', 'profit_before_tax', 'tax', 'net_profit', 'noplat'
]

df_quarterly_ttm = calculate_ttm_inplace(df_quarterly, ttm_columns)
df_quarterly_ttm = df_quarterly_ttm.dropna(subset=ttm_columns)

# Combine annual and quarterly data, excluding quarterly data if annual data exists for the same date and ticker
df_annual['record_type'] = 'Annual'
df_quarterly_ttm['record_type'] = 'Quarterly'

# Concatenate annual and quarterly data
df_combined = pd.concat([df_annual, df_quarterly_ttm], ignore_index=True)

# Drop the specified columns
df_combined.drop(columns=['ttm_sales_g', 'ttm_ebitda_g', 'ttm_opm', 'mcap'], inplace=True)

# Sort by date and ticker
df_combined.sort_values(by=['date', 'ticker'], inplace=True)

# Drop quarterly records if an annual record exists for the same ticker and date
df_combined = df_combined.drop_duplicates(subset=['ticker', 'date'], keep='first')

# List of columns to forward fill
# columns_to_fill = df_annual.columns[df_annual.columns.get_loc('eps'):].tolist()
columns_to_fill = [
    'eps', 'pe', 'ebitm', 'dividend_payout', 'opm', 'npm', 'roic', 'tax_rate', 'sales_g', 
    'ebitda_g', 'pat_g', 'non_op_inc_g', 'debt_to_capital', 'ppe_to_sales', 'dep_to_ppe', 
    'non_op_inc_to_invst', 'noplat', 'capex1', 'fcff', 'sales_g_3yr', 'sales_g_5yr', 
    'sales_g_10yr', 'ebitda_g_3yr', 'ebitda_g_5yr', 'ebitda_g_10yr', 'avg_opm_3yr', 
    'avg_opm_5yr', 'avg_opm_10yr', 'avg_opm', 'opm_min', 'opm_max', 'avg_roic_3yr', 
    'avg_roic_5yr', 'avg_roic_10yr', 'avg_roic', 'roic_min', 'roic_max', 'avg_ppe_to_sales', 
    'avg_dep_to_ppe', 'equity_share_capital', 'reserves', 'borrowings', 'other_liabilities', 
    'total_equity_and_debt', 'other_non_curr_asset', 'net_block', 'capital_work_in_progress', 
    'investments', 'other_assets', 'total_assets', 'capex2', 'working_capital', 'debtors', 
    'inventory', 'excess_cash', 'debtor_days', 'inventory_turnover', 'op_cash', 'total_cash', 
    'return_on_equity', 'return_on_capital_emp', 'cash_from_operating_activity', 
    'cash_from_investing_activity', 'cash_from_financing_activity', 'net_cashflow'
]

# Columns to forward fill from 'eps' onward
columns_to_fill = df_annual.columns[df_annual.columns.get_loc('eps'):].tolist()

# Forward fill missing values within each ticker group
df_combined = df_combined.groupby('ticker', group_keys=False).apply(lambda group: group.ffill())

# Create a dynamic filename using the subindustry and cap
filename = f'combined_annual_quarterly_data_{subindustry}_{cap}.csv'

# Save the combined DataFrame to a CSV file
df_combined.to_csv(filename, index=False)

print(f"Data saved to {filename}")

Data saved to combined_annual_quarterly_data_all_industry_all.csv


In [18]:
# Define the SQL query
query_daily_data = """
SELECT 
    date,
    name, 
    market_cap, 
    net_profit, 
    sales, 
    fcf_s, 
    yoy_quarterly_sales_growth, 
    yoy_quarterly_profit_growth, 
    qoq_sales_growth, 
    qoq_profit_growth, 
    opm_latest_quarter, 
    opm_last_year, 
    npm_latest_quarter, 
    npm_last_year, 
    profit_growth_3years, 
    sales_growth_3years, 
    sales_growth_5years, 
    sales_growth_10years, 
    pe_ttm, 
    historical_pe_3years, 
    peg_ratio, 
    pb_ttm, 
    ev_to_ebit, 
    dividend_payout, 
    roce, 
    roe, 
    avg_roce_3years, 
    avg_roe_3years, 
    debt, 
    debt_to_equity, 
    debt_3years_back, 
    mcap_to_netprofit, 
    mcap_to_sales, 
    noplat, 
    capex, 
    fcff, 
    invested_capital, 
    roic
FROM 
    daily_data_s
ORDER BY 
    date, name;
"""

# Fetch the data
try:
    connection = pymysql.connect(host=host, user=user, password=password, database=database)
    df_daily = pd.read_sql(query_daily_data, connection)
except pymysql.MySQLError as e:
    print(f"Error: {e}")
finally:
    if connection:
        connection.close()

query_traded_value = """
SELECT 
    date, 
    nse_ticker AS name, 
    total_traded_value
FROM 
    nse_price_history
ORDER BY 
    date, nse_ticker;
"""

# Fetch the data
try:
    connection = pymysql.connect(host=host, user=user, password=password, database=database)
    df_traded_value = pd.read_sql(query_traded_value, connection)
except pymysql.MySQLError as e:
    print(f"Error: {e}")
finally:
    if connection:
        connection.close()
# Merge the two DataFrames on 'date' and 'name'
df_daily = pd.merge(df_daily, df_traded_value, on=['date', 'name'], how='left')

# Create a dynamic filename
filename = f'daily_data_{subindustry}_{cap}.csv'

print(f"Data saved to {filename}")

Data saved to daily_data_all_industry_all.csv


In [20]:
# Assuming df_combined and df_daily are populated.

# Ensure df_daily['date'] and df_combined['result_date'] and df_combined['date'] are in pd.Timestamp format
df_daily['date'] = pd.to_datetime(df_daily['date'])
df_combined['date'] = pd.to_datetime(df_combined['date'])
df_combined['result_date'] = pd.to_datetime(df_combined['result_date'])

# Step 1: Replace result_date with date if result_date is null or before 2005-01-01
df_combined['result_date'] = df_combined.apply(
    lambda row: row['date'] if pd.isnull(row['result_date']) or row['result_date'] < pd.Timestamp('2005-01-01') else row['result_date'],
    axis=1
)

# Identify duplicates in df_combined based on 'ticker' and 'result_date'
duplicates_mask = df_combined.duplicated(subset=['ticker', 'result_date'], keep=False)

# For duplicates, replace 'result_date' with 'date'
df_combined.loc[duplicates_mask, 'result_date'] = df_combined.loc[duplicates_mask, 'date']

# Drop duplicates based on 'ticker' and 'result_date', keeping the first occurrence
# df_combined = df_combined.drop_duplicates(subset=['ticker', 'result_date'], keep='first')

print("df_combined size" , len(df_combined))
print("df_daily size" , len(df_daily))

# Function to adjust result_date only if it's greater than or equal to the minimum date for each ticker in df_daily
def adjust_to_next_available_date(result_date, daily_dates, min_daily_date):
    if result_date >= min_daily_date:
        if result_date not in daily_dates:
            # Find the next available date
            next_dates = [d for d in daily_dates if d > result_date]
            return next_dates[0] if next_dates else result_date
    return result_date

# Get unique daily dates for each ticker in df_daily
ticker_min_daily_dates = df_daily.groupby('name')['date'].min()
unique_daily_dates_by_ticker = {
    ticker: sorted(pd.to_datetime(group['date'].unique())) for ticker, group in df_daily.groupby('name')
}

# Apply the function to adjust result_date for each ticker
for ticker, daily_dates in unique_daily_dates_by_ticker.items():
    min_daily_date = ticker_min_daily_dates[ticker]
    df_combined.loc[df_combined['ticker'] == ticker, 'result_date'] = df_combined.loc[
        df_combined['ticker'] == ticker, 'result_date'
    ].apply(lambda date: adjust_to_next_available_date(date, daily_dates, min_daily_date))

# Ensure df_combined is sorted by 'date' and 'ticker'
df_combined.sort_values(by=['date', 'ticker'], inplace=True)

print("df_combined size" , len(df_combined))

# Perform the merge on df_daily['name'] == df_combined['ticker'] and df_daily['date'] == df_combined['result_date']
final_train_test = pd.merge(
    df_daily,
    df_combined,
    left_on=['name', 'date'],
    right_on=['ticker', 'result_date'],
    how='left',
    suffixes=('_daily', '_combined')
)
print("final_train_test size 1" , len(final_train_test))

# Drop redundant columns after the merge
final_train_test.drop(columns=['ticker', 'date_combined'], errors='ignore', inplace=True)

# Rename 'date_daily' to 'date' for consistency if it exists
final_train_test.rename(columns={'date_daily': 'date'}, inplace=True)

# Ensure the final DataFrame is sorted by 'date' and 'name'
final_train_test.sort_values(by=['date', 'name'], inplace=True)

# Reset index for cleanliness
final_train_test.reset_index(drop=True, inplace=True)

# List of columns to fill forward starting from 'sales_combined'
columns_to_fill = final_train_test.columns[final_train_test.columns.get_loc('sales_combined'):]

# Function to forward-fill missing values within each 'name' group
def fill_missing_values(group):
    # Track the last non-null row for the specified columns
    last_valid_values = None
    
    for i, row in group.iterrows():
        # Check if 'sales_combined' is non-null
        if pd.notnull(row['sales_combined']):
            last_valid_values = row[columns_to_fill]
        else:
            # If 'sales_combined' is null, fill with the last valid values if they exist
            if last_valid_values is not None:
                group.loc[i, columns_to_fill] = last_valid_values
                
    return group

# Apply the function to each 'name' group
final_train_test = final_train_test.groupby('name', group_keys=False).apply(fill_missing_values)

# Reset indices to ensure no indexing issues
final_train_test.reset_index(drop=True, inplace=True)
df_combined.reset_index(drop=True, inplace=True)

# Identify the columns to fill starting from 'sales_combined'
columns_to_fill = final_train_test.columns[final_train_test.columns.get_loc('sales_combined'):]

# Automatically identify columns with exact matches
common_columns = [col for col in columns_to_fill if col.replace('_combined', '') in df_combined.columns]

# Create a mapping for columns that do not match exactly
columns_mapping = {
    'sales_combined': 'sales',
    'net_profit_combined': 'net_profit',
    'market_cap_combined': 'market_cap',
    'dividend_payout_combined': 'dividend_payout',
    'roic_combined': 'roic',
    'noplat_combined': 'noplat',
    'fcff_combined': 'fcff'
}

# Create a lookup dictionary for each ticker in df_combined
combined_lookup = df_combined.groupby('ticker')

# Function to fill missing values within each 'name' group in final_train_test
def fill_group(group):
    ticker = group['name'].iloc[0]
    if ticker in combined_lookup.groups:
        combined_data = combined_lookup.get_group(ticker)
        for i, row in group.iterrows():
            if pd.isnull(row['sales_combined']):  # Only fill if 'sales_combined' is null
                valid_records = combined_data[combined_data['result_date'] <= row['date']]
                if not valid_records.empty:
                    recent_record = valid_records.sort_values(by='result_date').iloc[-1]
                    # Fill columns with exact matches
                    for col in common_columns:
                        group.at[i, col] = recent_record[col.replace('_combined', '')]
                    # Fill columns using the mapping
                    for final_col, combined_col in columns_mapping.items():
                        group.at[i, final_col] = recent_record[combined_col]
    return group

# Apply the function to each 'name' group
final_train_test = final_train_test.groupby('name', group_keys=False).apply(fill_group)

print("Filling missing values completed successfully.")

# Create a dynamic filename using the subindustry and cap
filename = f'final_train_test_{subindustry}_{cap}.csv'

# Save the final DataFrame to the CSV file with the dynamic filename
final_train_test.to_csv(filename, index=False)

print(f"Data saved to {filename}")

df_combined size 23890
df_daily size 967521
df_combined size 23890
final_train_test size 1 967678
Filling missing values completed successfully.
Data saved to final_train_test_all_industry_all.csv


In [ ]:
import pandas as pd

# Assuming final_train_test and df_combined are already defined

# Ensure date formats are consistent in both DataFrames
df_combined['result_date'] = pd.to_datetime(df_combined['result_date']).dt.normalize()
final_train_test['date'] = pd.to_datetime(final_train_test['date']).dt.normalize()

# Step 1: Identify records in df_combined that are not in final_train_test based on 'ticker' and 'result_date'
merged_keys = final_train_test[['name', 'date']].drop_duplicates()
remaining_df = df_combined[
    ~df_combined.set_index(['ticker', 'result_date']).index.isin(
        merged_keys.set_index(['name', 'date']).index
    )
]

# Step 2: Filter out records where market_cap is null or <= 0
remaining_df = remaining_df[remaining_df['market_cap'].notnull() & (remaining_df['market_cap'] > 0)]

# Step 3: Assign values to daily columns based on earlier mappings
remaining_df['name'] = remaining_df['ticker']
remaining_df['date'] = remaining_df['result_date']

# Create a copy of remaining_df to ensure no SettingWithCopyWarning
remaining_df_final = remaining_df.copy()

# Explicitly set the columns as needed
remaining_df_final['sales_combined'] = remaining_df_final['sales']
remaining_df_final['net_profit_combined'] = remaining_df_final['net_profit']
remaining_df_final['market_cap_combined'] = remaining_df_final['market_cap']
remaining_df_final['dividend_payout_combined'] = remaining_df_final['dividend_payout']
remaining_df_final['roic_combined'] = remaining_df_final['roic']
remaining_df_final['noplat_combined'] = remaining_df_final['noplat']
remaining_df_final['fcff_combined'] = remaining_df_final['fcff']

# Assigning values to the corresponding daily columns explicitly
remaining_df_final['market_cap_daily'] = remaining_df_final['market_cap']
remaining_df_final['net_profit_daily'] = remaining_df_final['net_profit']
remaining_df_final['sales_daily'] = remaining_df_final['sales']
remaining_df_final['fcf_s'] = remaining_df_final['fcff']
remaining_df_final['yoy_quarterly_sales_growth'] = remaining_df_final['sales_g']
remaining_df_final['yoy_quarterly_profit_growth'] = remaining_df_final['pat_g']
remaining_df_final['qoq_sales_growth'] = remaining_df_final['sales_g']
remaining_df_final['qoq_profit_growth'] = remaining_df_final['pat_g']
remaining_df_final['opm_latest_quarter'] = remaining_df_final['opm']
remaining_df_final['opm_last_year'] = remaining_df_final['avg_opm_3yr']
remaining_df_final['npm_latest_quarter'] = remaining_df_final['npm']
remaining_df_final['npm_last_year'] = remaining_df_final['avg_opm_3yr']
remaining_df_final['profit_growth_3years'] = remaining_df_final['ebitda_g_3yr']
remaining_df_final['sales_growth_3years'] = remaining_df_final['sales_g_3yr']
remaining_df_final['sales_growth_5years'] = remaining_df_final['sales_g_5yr']
remaining_df_final['sales_growth_10years'] = remaining_df_final['sales_g_10yr']
remaining_df_final['pe_ttm'] = remaining_df_final['pe']
remaining_df_final['historical_pe_3years'] = remaining_df_final['pe']
remaining_df_final['peg_ratio'] = 0
remaining_df_final['pb_ttm'] = 0
remaining_df_final['ev_to_ebit'] = 0
remaining_df_final['dividend_payout_daily'] = remaining_df_final['dividend_payout']
remaining_df_final['roce'] = remaining_df_final['roic']
remaining_df_final['roe'] = 0
remaining_df_final['avg_roce_3years'] = 0
remaining_df_final['avg_roe_3years'] = 0
remaining_df_final['debt'] = remaining_df_final['borrowings']
remaining_df_final['debt_to_equity'] = remaining_df_final['debt_to_capital']
remaining_df_final['debt_3years_back'] = remaining_df_final['borrowings']
remaining_df_final['mcap_to_netprofit'] = remaining_df_final['pe']
remaining_df_final['mcap_to_sales'] = 0
remaining_df_final['noplat_daily'] = remaining_df_final['noplat']
remaining_df_final['capex'] = remaining_df_final['capex1']
remaining_df_final['fcff_daily'] = remaining_df_final['fcff']
remaining_df_final['invested_capital'] = 0
remaining_df_final['roic_daily'] = remaining_df_final['roic']
remaining_df_final['total_traded_value'] = 0

# Step 4: Add missing columns with default values (None) to match final_train_test
for col in final_train_test.columns:
    if col not in remaining_df_final.columns:
        remaining_df_final[col] = None

# Step 5: Ensure the column order matches final_train_test
remaining_df_final = remaining_df_final[final_train_test.columns]

# Step 6: Concatenate the remaining_df_final to final_train_test
final_train_test = pd.concat([final_train_test, remaining_df_final], ignore_index=True)

# Step 7: Sort the final DataFrame by 'date' and 'name'
final_train_test.sort_values(by=['date', 'name'], inplace=True)
final_train_test.reset_index(drop=True, inplace=True)

# Step 8: Save the final DataFrame to a CSV file with a dynamic filename
filename = f'final_train_test_{subindustry}_{cap}.csv'
final_train_test.to_csv(filename, index=False)
print(f"Data saved to {filename}")

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import os

# Flag to decide if training should be done from scratch
train_from_scratch = True

# Model file path (dynamic based on subindustry and cap)
model_path = f'xgboost_marketcap_model_{subindustry}_{cap}.pkl'

# Logging function
def log(message):
    print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - {message}")

# Step 1: Create the target variable (1-year ahead market cap)
log("Creating 1-year ahead market cap target...")
def get_one_year_ahead_market_cap(df):
    # Ensure data is sorted by name and date
    df = df.sort_values(by=['name', 'date'])

    # Create a new column for the target date (exactly 1 year ahead)
    df['target_date'] = df['date'] + pd.DateOffset(years=1)

    # Self-join to match each row with its corresponding target date
    df_with_target = df.merge(
        df[['name', 'date', 'market_cap_daily']],
        left_on=['name', 'target_date'],
        right_on=['name', 'date'],
        suffixes=('', '_target'),
        how='left'
    )

    # Rename the merged market cap column to 'target_market_cap'
    df_with_target.rename(columns={'market_cap_daily_target': 'target_market_cap'}, inplace=True)

    # Drop the duplicate 'date' column from the merged result
    df_with_target.drop(columns=['date_target'], inplace=True)

    # Drop rows where 'target_market_cap' is missing
    df_with_target = df_with_target.dropna(subset=['target_market_cap'])

    return df_with_target

# Load final_train_test DataFrame (assuming it's already defined)
final_train_test = get_one_year_ahead_market_cap(final_train_test)
log("Target variable created.")

# Step 2: Feature Engineering (excluding non-numeric and target columns)
log("Preparing features and target...")
features = final_train_test.drop(columns=[
    'date', 'name', 'market_cap_daily', 'result_date', 'target_date', 'record_type'
])

features_sorted = features.sort_values(by='date')
target_sorted = features_sorted['target_market_cap']
features_sorted.drop(columns=['target_market_cap'])
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(features_sorted, target_sorted, test_size=0.3, shuffle=False)

log("Data split completed.")


# Check if the model already exists and the flag for training
if os.path.exists(model_path) and not train_from_scratch:
    log("Loading existing model...")
    best_model = joblib.load(model_path)
    log("Model loaded successfully.")
else:
    # Step 4: Define XGBoost model and GridSearchCV parameters
    log("Training the model with GridSearchCV...")
    xgb_model = XGBRegressor(objective='reg:squarederror', random_state=42)

    param_grid = {
        'n_estimators': [100, 200, 300, 500],
        'max_depth': [3, 5, 7, 9],
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'subsample': [0.6, 0.8, 1.0],
        'colsample_bytree': [0.6, 0.8, 1.0]
    }

    grid_search = GridSearchCV(estimator=xgb_model, param_grid=param_grid, cv=3, scoring='r2', verbose=2, n_jobs=-1)

    # Measure training time
    start_time = datetime.now()
    grid_search.fit(X_train, y_train)
    end_time = datetime.now()
    log(f"Model training completed in {end_time - start_time}.")

    # Best parameters from GridSearchCV
    log(f"Best Parameters: {grid_search.best_params_}")

    # Save the best model
    best_model = grid_search.best_estimator_
    joblib.dump(best_model, model_path)
    log(f"Best model saved as {model_path}.")

# Step 5: Evaluate the model on the test set
log("Evaluating the model on the test set...")
start_time = datetime.now()
y_pred = best_model.predict(X_test)
end_time = datetime.now()
log(f"Model testing completed in {end_time - start_time}.")

# Calculate performance metrics
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

log(f"Test RMSE: {rmse:.2f}")
log(f"Test R²: {r2:.2f}")

# Optional: Feature Importance
importances = best_model.feature_importances_
feature_names = features.columns
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances}).sort_values(by='Importance', ascending=False)

log("Feature Importances:")
print(feature_importance_df)

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

# Logging function
def log(message):
    print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - {message}")

# Example ticker
ticker = 'ANUP'

# Path to the CSV file used for training
csv_path = filename

# Option to specify the latest date (set to '' to fetch the latest date automatically)
latest_date_input = ''  # e.g., '2024-12-13' or leave as '' for the latest date

def get_latest_data_and_predict_from_csv(ticker, csv_path, model, latest_date_input=''):
    log(f"Fetching data for ticker: {ticker} from {csv_path}")

    # Load the CSV data
    df = pd.read_csv(csv_path)

    # Convert 'date' to datetime
    df['date'] = pd.to_datetime(df['date'], errors='coerce')

    # Filter data for the specific ticker
    ticker_data = df[df['name'] == ticker]

    if ticker_data.empty:
        log(f"No data found for ticker: {ticker}")
        return

    # Determine the row based on the specified latest date or use the max date if not specified
    if latest_date_input:
        try:
            specified_date = pd.to_datetime(latest_date_input)
            latest_row = ticker_data[ticker_data['date'] == specified_date]
            if latest_row.empty:
                log(f"No data found for ticker: {ticker} on specified date: {latest_date_input}")
                return
            latest_row = latest_row.iloc[0]
        except Exception as e:
            log(f"Error parsing the specified date: {latest_date_input}. Error: {e}")
            return
    else:
        # Use the latest available date
        latest_row = ticker_data.loc[ticker_data['date'].idxmax()]

    latest_date = latest_row['date']
    current_market_cap = latest_row['market_cap_daily']

    log(f"Latest available date for {ticker}: {latest_date.strftime('%Y-%m-%d')}")

    # Prepare the test data by dropping unnecessary columns
    columns_to_drop = ['date', 'name', 'market_cap_combined', 'sector', 'industry', 
                       'sub_industry', 'target_date', 'record_type', 'result_date']
    test_data = latest_row.drop(labels=columns_to_drop, errors='ignore')

    # Convert to DataFrame with a single row and ensure numeric data
    test_data_df = pd.DataFrame([test_data]).apply(pd.to_numeric, errors='coerce')

    # Predict the target market cap
    predicted_market_cap = float(model.predict(test_data_df.fillna(0))[0])

    # Calculate expected percentage change
    expected_change = ((predicted_market_cap / current_market_cap) - 1) * 100

    # Print the results
    print(f"\nPrediction Results for {ticker}")
    print(f"Latest Available Date: {latest_date.strftime('%Y-%m-%d')}")
    print(f"Current Market Cap: {current_market_cap:,.2f}")
    print(f"Predicted Target Market Cap (1 year ahead): {predicted_market_cap:,.2f}")
    print(f"Expected % Change Over the Year: {expected_change:.2f}%")

# Call the function
get_latest_data_and_predict_from_csv(ticker, csv_path, best_model, latest_date_input)